#Project


#Code to load the datasets

In [ ]:
# Data setup: download datasets from shared Google Drive

!pip -q install gdown

import os
from pathlib import Path

DATA_DIR = Path("/content/ML_Exam_Project")
DATA_DIR.mkdir(exist_ok=True)

FOLDER_URL = "https://drive.google.com/drive/folders/1f8t9t6jLE6vFpnu7A-xQuhieDlhk48Iw?usp=share_link"

!gdown --folder "$FOLDER_URL" -O "$DATA_DIR"

Retrieving folder contents
Processing file 1uSTcAolrashExIz6RzS7xMTdiZY7xsxF test_id.jsonl
Processing file 1-y5suAJGJK-OgY7yNm9LkiWnFmCV0zj7 test_long.jsonl
Processing file 1Gr1newDY91vYEQjdyzshR5LYSU_VqT17 test_ood.jsonl
Processing file 184ksDE_EtHRYNt2FaP4Qvhw85OA1AAR4 train.jsonl
Processing file 1nRoACGPmX7BT9vlG_65aenEiXDJmNlwj validation.jsonl
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1uSTcAolrashExIz6RzS7xMTdiZY7xsxF
To: /content/ML_Exam_Project/test_id.jsonl
100% 222k/222k [00:00<00:00, 113MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-y5suAJGJK-OgY7yNm9LkiWnFmCV0zj7
To: /content/ML_Exam_Project/test_long.jsonl
100% 182k/182k [00:00<00:00, 56.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Gr1newDY91vYEQjdyzshR5LYSU_VqT17
To: /content/ML_Exam_Project/test_ood.jsonl
100% 230k/230k [00:00<00:00, 95.4MB/s]
Downloading...
From: https://drive.goo

In [ ]:
# Check

from pathlib import Path

DATA_DIR = Path("/content/ML_Exam_Project")

expected_files = [
    "train.jsonl",
    "validation.jsonl",
    "test_id.jsonl",
    "test_ood.jsonl",
    "test_long.jsonl",
]

for filename in expected_files:
    path = DATA_DIR / filename
    assert path.exists(), f"Missing file: {path}"

print("All dataset files found:")
for filename in expected_files:
    print(" -", DATA_DIR / filename)

All dataset files found:
 - /content/ML_Exam_Project/train.jsonl
 - /content/ML_Exam_Project/validation.jsonl
 - /content/ML_Exam_Project/test_id.jsonl
 - /content/ML_Exam_Project/test_ood.jsonl
 - /content/ML_Exam_Project/test_long.jsonl


In [ ]:
import json
import pandas as pd

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return pd.DataFrame(records)

In [ ]:
# Load the datasets

train_df = load_jsonl(DATA_DIR / "train.jsonl")
validation_df = load_jsonl(DATA_DIR / "validation.jsonl")
test_id_df = load_jsonl(DATA_DIR / "test_id.jsonl")
test_ood_df = load_jsonl(DATA_DIR / "test_ood.jsonl")
test_long_df = load_jsonl(DATA_DIR / "test_long.jsonl")

x_train = train_df["expression"]
y_train = train_df["value"]
x_val = validation_df["expression"]
y_val = validation_df["value"]

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test ID:", test_id_df.shape)
print("Test OOD:", test_ood_df.shape)
print("Test long:", test_long_df.shape)

train_df.head()

Train: (12000, 6)
Validation: (2000, 6)
Test ID: (2000, 6)
Test OOD: (2000, 6)
Test long: (1500, 6)


,id,expression,value,length,operator_count,depth
0,train-00000,4+4+7+3+8-1,25,11,5,5
1,train-00001,5-(6-5)+8+8,20,11,4,5
2,train-00002,5-3+1,3,5,2,3
3,train-00003,1-(1+5+8+7+1),-21,13,5,6
4,train-00004,7+1-(6+5+9),-12,11,4,4


##Tokenization

In [ ]:
import random
import math
import copy
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
all_chars = set()                   #Counting all the characters in every training expression and adding it to all_chars if new
for expression in x_train:
    for char in expression:
        all_chars.add(char)


V = sorted(list(all_chars))
PAD_TOKEN = "[PAD]"
V = [PAD_TOKEN] + V
print(len(V))
token_to_index = {token: i for i, token in enumerate(V)}
print(token_to_index)

15
{'[PAD]': 0, '(': 1, ')': 2, '+': 3, '-': 4, '0': 5, '1': 6, '2': 7, '3': 8, '4': 9, '5': 10, '6': 11, '7': 12, '8': 13, '9': 14}


##CNN

In [ ]:
def encode(x):
    return [token_to_index[token] for token in list(x)]

def decode(x):
    return ''.join([V[index] for index in x if V[index] != PAD_TOKEN])

print(encode('+'))
print(decode(encode('+')))

[3]
+


In [ ]:
def collate_fn(batch):                #we dynamically pad every batch that has to be given to the nn by the maximum element in it
    # 'batch' is a list of (input_sequence, target_value) tuples
    # For us, input_sequence is x_data, and target_value is y_data

    # Separate inputs (expressions) and targets (values)
    expressions = [item[0] for item in batch]
    values = [item[1] for item in batch]

    # Encode expressions
    encoded_expressions = [encode(expr) for expr in expressions]

    # Find the maximum length in the current batch
    batch_max_length = max(len(seq) for seq in encoded_expressions)

    # Pad each sequence in the batch to batch_max_length
    padded_expressions = []
    for seq in encoded_expressions:
        padding_needed = batch_max_length - len(seq)
        padded_seq = seq + [token_to_index[PAD_TOKEN]] * padding_needed
        padded_expressions.append(padded_seq)

    # Convert to PyTorch tensors
    x_batch = torch.tensor(padded_expressions, dtype=torch.long)
    y_batch = torch.tensor(values, dtype=torch.float)

    return x_batch, y_batch

In [ ]:
VOCAB_SIZE = len(V)
#EMBEDDING_DIM = 64 # Dimension for token embeddings
NUM_CLASSES = 1 # For regression, we predict a single value (the result of the expression)

#THIS IS GIVEN BY CHATGPT

import torch.nn as nn
import torch.nn.functional as F

class ExpressionCNNOneHot(nn.Module):                               #we are defining a new model to enable the dynamic padding of the inputs
    def __init__(self, vocab_size, hidden_channels=64, num_outputs=1):            #num_outputs is for classification against regression problems, 199 for classification, 1 for regression
        super(ExpressionCNNOneHot, self).__init__()                         #all the attributes of the nn.Module class, (we need to see it a little bit)

        self.conv1 = nn.Conv1d(                                         #in this type of model we add 1 convolution, the inputs have the vocab size has dimension, the outputs have hidden layers
            in_channels=vocab_size,
            out_channels=hidden_channels,
            kernel_size=3,
            padding=1
        )

        self.conv2 = nn.Conv1d(
            in_channels=hidden_channels,
            out_channels=hidden_channels,
            kernel_size=3,
            padding=1
        )

        self.fc = nn.Linear(hidden_channels, num_outputs)

    def forward(self, x, mask):
        # x shape: (batch_size, vocab_size, sequence_length)
        # mask shape: (batch_size, sequence_length)

        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))

        # Masked mean pooling
        # h shape: (batch_size, hidden_channels, sequence_length)
        mask = mask.unsqueeze(1)  # (batch_size, 1, sequence_length)

        h = h * mask

        lengths = mask.sum(dim=2).clamp(min=1)  # (batch_size, 1)

        pooled = h.sum(dim=2) / lengths  # (batch_size, hidden_channels)

        output = self.fc(pooled).squeeze(1)

        return output

In [ ]:
model = ExpressionCNNOneHot(
    vocab_size=VOCAB_SIZE,
    hidden_channels=64,
    num_outputs=1
)

###Training the model

In [ ]:
def train(model, x_train_data, y_train_data, optimizer, loss_fn, epochs, device, batch_size, x_val_data=None, y_val_data=None):
    model.to(device)
    # Create DataLoader for training data
    train_dataset = list(zip(x_train_data, y_train_data))
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

    # Create DataLoader for validation data if provided
    val_loader = None
    if x_val_data is not None and y_val_data is not None:
        val_dataset = list(zip(x_val_data, y_val_data))
        val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    for epoch in range(epochs):
        model.train()
        losses = []
        for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            mask = (x_batch != token_to_index[PAD_TOKEN]).float()

            # One-hot encode x_batch and permute for conv1d
            x_batch = F.one_hot(x_batch, num_classes=VOCAB_SIZE).float().permute(0, 2, 1)

            optimizer.zero_grad()
            y_pred = model(x_batch, mask)
            loss = loss_fn(y_pred, y_batch)

            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        train_loss = np.mean(losses)
        print(f'Train Epoch: {epoch + 1}, Train Loss = {train_loss:.4f}')

        if val_loader is not None:
            val_loss = evaluate(model, val_loader, loss_fn, device)
            print(f'Validation Loss: {val_loss:.4f}')

def evaluate(model, loader, loss_fn, device):
    model.to(device)
    model.eval()
    total_loss = 0
    num_samples = 0
    # The 'correct' variable is not used in a regression context, so it will be removed.
    # correct = 0 # No longer relevant for regression metrics

    with torch.no_grad():
        for batch_size, (x_batch, y_batch) in enumerate(loader):
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            mask = (x_batch != token_to_index[PAD_TOKEN]).float()

            # One-hot encode x_batch and permute for conv1d
            x_batch = F.one_hot(x_batch, num_classes=VOCAB_SIZE).float().permute(0, 2, 1)

            y_pred = model(x_batch, mask)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item() * x_batch.size(0)
            num_samples += x_batch.size(0)
            # The following line is for classification accuracy and is not relevant for regression.
            # correct += y_pred.eq(y_batch.view_as(y_pred)).sum().item()

    # The 'accuracy' calculation is for classification and will be removed.
    # accuracy = correct/num_samples
    average_loss = total_loss / num_samples
    # Updated print statement for regression, removing accuracy.
    print(f'Validation set: Average loss: {average_loss:.4f}')
    return average_loss

In [ ]:
batch_size = 32
learning_rate = 0.001
epochs = 40
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)
loss_fn = nn.MSELoss() # Changed from CrossEntropyLoss to MSELoss for regression
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train(model, x_train, y_train, optimizer, loss_fn, epochs, device, batch_size, x_val, y_val)          #it is normal for regression, we should normalize the datas and the rescale them, the important thing is that it is decreasing

Train Epoch: 1, Train Loss = 60903654.7200
Validation set: Average loss: 31226504.9920
Validation Loss: 31226504.9920
Train Epoch: 2, Train Loss = 18526376.7920
Validation set: Average loss: 10856440.7680
Validation Loss: 10856440.7680
Train Epoch: 3, Train Loss = 7395042.5727
Validation set: Average loss: 5420991.3440
Validation Loss: 5420991.3440
Train Epoch: 4, Train Loss = 3887857.6857
Validation set: Average loss: 2992714.0640
Validation Loss: 2992714.0640
Train Epoch: 5, Train Loss = 2155477.7113
Validation set: Average loss: 1686456.1890
Validation Loss: 1686456.1890
Train Epoch: 6, Train Loss = 1203054.1911
Validation set: Average loss: 941072.8545
Validation Loss: 941072.8545
Train Epoch: 7, Train Loss = 664424.1208
Validation set: Average loss: 521771.4917
Validation Loss: 521771.4917
Train Epoch: 8, Train Loss = 366663.7349
Validation set: Average loss: 291298.5634
Validation Loss: 291298.5634
Train Epoch: 9, Train Loss = 209235.3267
Validation set: Average loss: 168766.7176